Screenshots of the Lab 2 part 1:

Imports:

In [0]:
from pyspark.sql.functions import input_file_name, current_timestamp, current_date
import sys
import re

In [0]:
# path to the tables
source_path = "/Volumes/dbr_dev/konstyantyn_bronze/raw_data/"
checkpoint_path = "/Volumes/dbr_dev/konstyantyn_bronze/raw_data/_checkpoints/bronze_table/"
schema_location = "/Volumes/dbr_dev/konstyantyn_bronze/raw_data/_schemas/bronze_table/"
target_table = "dbr_dev.konstyantyn_bronze.bronze_data"

# Enable Column Mapping in the current Spark session to support spaces, brackets, and special characters
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# try function to test code for mistakes
try:
    # check if the source path exists and what format it has csv or json
    files = dbutils.fs.ls(source_path)
    if len(files) == 0:
        raise Exception("No files found in the source path")
    else:
        print("Files found in the source path")
        
    format_founded = None
    for file in files:
        if file.name.endswith(".csv"):
            format_founded = "csv"
            break
        elif file.name.endswith(".json"):
            format_founded = "json"
            break
            
    if format_founded is None:
        raise Exception("No supported file (CSV, JSON) format found in the source path check if they hafe different type(txt, parquet, etc.)")
    else:
        print(f"Supported file format found in the source path: {format_founded}")
        
    # check if csv file or json
    if format_founded == "csv":
        df_raw = (spark.readStream
          .format("cloudFiles")
          .option("cloudFiles.format", "csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .option("cloudFiles.schemaLocation", schema_location)
          .load(source_path)
        )
    elif format_founded == "json":
        df_raw = (spark.readStream
          .format("cloudFiles")
          .option("cloudFiles.format", "json")
          .option("cloudFiles.inferColumnTypes", "true")
          .option("cloudFiles.schemaLocation", schema_location)
          .load(source_path)
        )
        
    # Advanced Column Cleaning: Fix empty names and replace unsafe characters
    for idx, col_name in enumerate(df_raw.columns):
        # Replace all spaces, slashes, periods, brackets, and dashes with underscores
        clean_name = re.sub(r'[ ,;{}()\n\t=\.\/-]', '_', col_name)
        # Remove multiple consecutive underscores (e.g., ___ becomes _)
        clean_name = re.sub(r'_{2,}', '_', clean_name).strip('_')
        
        # If the name became empty or invalid, assign a unique default name
        if not clean_name or clean_name == "":
            clean_name = f"unnamed_col_{idx}"
            
        df_raw = df_raw.withColumnRenamed(col_name, clean_name)
        
    # add columns to the dataframe
    df_bronze = (df_raw
      .withColumn("source_filename", df_raw["_metadata.file_path"])
      .withColumn("ingestion_timestamp", current_timestamp())
      .withColumn("load_date", current_date())
    )
    
    print(f"Writing data to the table: {target_table}...")
    
    # write data to the bronze table
    query = (df_bronze.writeStream
      .format("delta")
      .outputMode("append")
      .option("checkpointLocation", checkpoint_path)
      .option("delta.columnMapping.mode", "name")
      .toTable(target_table)
    )
    
    query.awaitTermination(timeout=30)
    print(f"Data written to the table: {target_table}")
    
except Exception as e:
    print(f"Error happend: {e}")

Files found in the source path
Supported file format found in the source path: csv
Writing data to the table: dbr_dev.konstyantyn_bronze.bronze_data...


In [0]:
#data check
df_check = spark.read.table("dbr_dev.konstyantyn_bronze.bronze_data")
print(f"Raw amount: {df_check.count()}")
display(df_check.limit(10))

Raw amount: 7884


Entity,Code,Year,Deaths_in_ongoing_conflicts_best_estimate_Conflict_type:_all,unnamed_col_4,rescued_data,source_filename,ingestion_timestamp,load_date
Abkhazia,OWID_ABK,1989,0,null,null,/Volumes/dbr_dev/konstyantyn_bronze/raw_data/deaths-in-armed-conflicts-based-on-where-they-occurred.csv,2026-07-12T18:27:22.086Z,2026-07-12
Abkhazia,OWID_ABK,1990,0,null,null,/Volumes/dbr_dev/konstyantyn_bronze/raw_data/deaths-in-armed-conflicts-based-on-where-they-occurred.csv,2026-07-12T18:27:22.086Z,2026-07-12
Abkhazia,OWID_ABK,1991,0,null,null,/Volumes/dbr_dev/konstyantyn_bronze/raw_data/deaths-in-armed-conflicts-based-on-where-they-occurred.csv,2026-07-12T18:27:22.086Z,2026-07-12
Abkhazia,OWID_ABK,1992,0,null,null,/Volumes/dbr_dev/konstyantyn_bronze/raw_data/deaths-in-armed-conflicts-based-on-where-they-occurred.csv,2026-07-12T18:27:22.086Z,2026-07-12
Abkhazia,OWID_ABK,1993,0,null,null,/Volumes/dbr_dev/konstyantyn_bronze/raw_data/deaths-in-armed-conflicts-based-on-where-they-occurred.csv,2026-07-12T18:27:22.086Z,2026-07-12
Abkhazia,OWID_ABK,1994,0,null,null,/Volumes/dbr_dev/konstyantyn_bronze/raw_data/deaths-in-armed-conflicts-based-on-where-they-occurred.csv,2026-07-12T18:27:22.086Z,2026-07-12
Abkhazia,OWID_ABK,1995,0,null,null,/Volumes/dbr_dev/konstyantyn_bronze/raw_data/deaths-in-armed-conflicts-based-on-where-they-occurred.csv,2026-07-12T18:27:22.086Z,2026-07-12
Abkhazia,OWID_ABK,1996,0,null,null,/Volumes/dbr_dev/konstyantyn_bronze/raw_data/deaths-in-armed-conflicts-based-on-where-they-occurred.csv,2026-07-12T18:27:22.086Z,2026-07-12
Abkhazia,OWID_ABK,1997,0,null,null,/Volumes/dbr_dev/konstyantyn_bronze/raw_data/deaths-in-armed-conflicts-based-on-where-they-occurred.csv,2026-07-12T18:27:22.086Z,2026-07-12
Abkhazia,OWID_ABK,1998,0,null,null,/Volumes/dbr_dev/konstyantyn_bronze/raw_data/deaths-in-armed-conflicts-based-on-where-they-occurred.csv,2026-07-12T18:27:22.086Z,2026-07-12
